In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from lightgbm import LGBMClassifier
import warnings

warnings.filterwarnings('ignore')

In [21]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [22]:
df = df.drop(["customerID"], axis=1)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [25]:
df["TotalCharges"].value_counts()

TotalCharges
          11
20.2      11
19.75      9
20.05      8
19.9       8
          ..
6849.4     1
692.35     1
130.15     1
3211.9     1
6844.5     1
Name: count, Length: 6531, dtype: int64

In [26]:
df.drop(df[df["tenure"] == 0].index, axis=0, inplace=True)

In [27]:
df[df["tenure"] == 0].index

Index([], dtype='int64')

In [28]:
df["TotalCharges"].value_counts()

TotalCharges
20.2      11
19.75      9
20.05      8
19.9       8
19.65      8
          ..
6849.4     1
692.35     1
130.15     1
3211.9     1
6844.5     1
Name: count, Length: 6530, dtype: int64

In [7]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1:"Yes"})

In [8]:
def object_to_int(dataframe_series):
    if dataframe_series.dtype == "object":
        dataframe_series = LabelEncoder().fit_transform(dataframe_series)
    return dataframe_series

df = df.apply(lambda x: object_to_int(x))

In [9]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [10]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [11]:
lgbm_model = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)

In [12]:
param_dist_lgbm = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [15, 31, 50, 63],
    'max_depth': [3, 5, 7, -1],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.8, 1.0],         
    'colsample_bytree': [0.6, 0.8, 1.0]   
}

In [14]:
print("LightGBM hiperparametre optimizasyonu başlatılıyor...")
random_lgbm = RandomizedSearchCV(
    estimator=lgbm_model, 
    param_distributions=param_dist_lgbm, 
    n_iter=30, 
    cv=5, 
    scoring="recall", 
    random_state=42, 
    n_jobs=-1, 
    verbose=1
)

random_lgbm.fit(X_train, y_train)
y_pred_lgbm = random_lgbm.predict(X_test)

print(f"\nLightGBM En İyi Recall Skoru: {random_lgbm.best_score_:.4f}")
print(f"LightGBM En İyi Parametreler: {random_lgbm.best_params_}\n")
print("--- Test Seti Performansı ---")
print(classification_report(y_test, y_pred_lgbm))

LightGBM hiperparametre optimizasyonu başlatılıyor...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

LightGBM En İyi Recall Skoru: 0.8107
LightGBM En İyi Parametreler: {'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 30, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 1.0}

--- Test Seti Performansı ---
              precision    recall  f1-score   support

           0       0.91      0.69      0.78      1033
           1       0.48      0.81      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.69      1407
weighted avg       0.80      0.72      0.74      1407

